# Interaction-Ready Synthetic Data — Live Workshop

**Real2Sim for Embodied Data Generation**
*Generating Interaction-Ready Data for Long-Horizon Articulated Manipulation*

Hardware path: a single **CDNA3 (MI300/MI325, gfx942)** path. Scope: **interaction-ready data generation only** —
we use CAP to declare a long-horizon articulated task `open → pick → place → close` (a drawer), and run it through rocRobo collision-free planning
to produce a LeRobot dataset with interaction semantics. This session does **not** cover train / eval.

> Just run the cells in order. The image ships with everything preinstalled; artifacts land in `output/`.

## 0. Environment check

Confirm two things are ready: ① this container can see the gfx942 GPU; ② the rocRobo collision-free planning sidecar (`rocrobo_dev`) is reachable.
(Both images are started ahead of time by the instructor.)

In [ ]:
import os, subprocess, torch

print("== GPU / ROCm ==")
print("torch:", torch.__version__, "| GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
print("HSA_OVERRIDE_GFX_VERSION:", os.environ.get("HSA_OVERRIDE_GFX_VERSION"))
print("ROCROBO_BACKEND:", os.environ.get("ROCROBO_BACKEND"))

print("\n== rocRobo sidecar ==")
r = subprocess.run(
    ["docker", "exec", "rocrobo_dev", "python", "-c", "import rocrobo; print('rocrobo_ok')"],
    capture_output=True, text=True,
)
print(r.stdout.strip() or r.stderr.strip() or "sidecar unreachable")

## 1. Where the last session ended → what this session adds

The [previous session](https://github.com/PhysicalAI-AIM/Robot_synthetic_data_generation_workshop/tree/main)
already proved the full chain **synthetic data → VLA training → closed-loop eval** runs end-to-end on AMD ROCm,
but the task was **grasping a single red cube** — a single-step rigid pick, where interaction is trivial.

This session adds just one step up: **can we generate data that "interacts with the world"**.

| Dimension | Last session (cube pick) | This session |
|---|---|---|
| Task | Single-step cube grasp | **Long-horizon `open→pick→place→close`** |
| Object | Pure rigid body | **Articulated body (drawer, URDF joint)** |
| Motion | IK + straight line | **Collision-free planning + per-segment re-anchoring + controlled contact phase** |
| Data semantics | state/action + images | **+ joint opening in obs, per-segment re-anchor, contact phase** |

## 2. Where the interaction task comes from: Real2Sim assets

Interaction data requires **interactable sim assets**: real objects/scenes → meshes / URDF (with joints, contact surfaces).
The asset pipeline goes through rocRecon (real → simulatable assets); this session falls back to built-in objaverse assets
(a drawer cabinet + a die cube). The key is the drawer's **URDF joint** — it makes "open / close" an interactable action.

## 3. CAP: declare an interaction task in ~15 lines

The scenario below is the entire author-side input. CAP decouples **task intent** from **execution details** —
the author declares just four layers:

- **scene**: which objects to place (drawer + die), anchors, cameras
- **target**: the goal point for placing into the drawer (anchored to the drawer's moving joint, so it follows the actual opening)
- **task**: natural-language instruction + success criterion (die inside the drawer AND drawer closed)
- **intents**: the skill sequence `open → pick → place → close`

Each participant declares their own interaction task by editing just this one file — this is the entry point of a "declarative data factory".

> **How do you write a CAP?** In this session you edit this `.py` directly in the Jupyter file editor — it lives in the repo mounted into the container,
> so saving takes effect in the container immediately, no need to "move files". CAP is deliberately kept to ~15 lines of readable Python, **edit it by hand, no local code agent required**;
> auto-generating CAP with an LLM/agent is the advanced code-as-policy play, a separate author-side track, not the live mechanism.

In [ ]:
from pathlib import Path

SCENARIO = "scenarios/pick_place_into_drawer.py"
print(Path(SCENARIO).read_text())

### Render the declaration into an image (check the scene first)

Before running the full rollout, **statically render one frame** of the scenario you just declared: build the scene + place the objects + reset Franka,
to confirm the drawer / die placement matches expectations. This step also pays off the first-time kernel compilation of `scene.build()` (now cached),
so the real generation later is faster.

In [ ]:
import subprocess, sys, glob
from pathlib import Path
from IPython.display import Image as IPyImage

# Follows SCENARIO above: whichever scenario you edited is the one previewed.
SNAP_OUT = f"output/{Path(SCENARIO).stem}_snapshot"
cmd = [
    sys.executable, "scripts/datagen/snapshot_scenario.py",
    "--scenario", SCENARIO,
    "--out", SNAP_OUT,
    "--seed", "42",
]
print(">>", " ".join(cmd), "\n")
subprocess.run(cmd, check=True)

pngs = sorted(glob.glob(f"{SNAP_OUT}/*overview*.png"))
IPyImage(filename=pngs[0]) if pngs else None

### 🙋 Your turn (second-level feedback, no need to wait for generation)

The snapshot only builds + renders one frame, so it produces an image in seconds. Now **you declare**: double-click to open
`scenarios/pick_place_into_drawer.py` in the left file tree, change one line and save, then **re-run the snapshot cell above** to see the change.

A few things to try (pick any):

- Change the `die`'s `y` in `fixed_position` (`0.32`) to `-0.15` → the cube moves to the other side of the drawer
- Replace `asset="die_01"` with `asset="apple_01"` → swap in a different object
- Change the `SCENARIO` above to `scenarios/pick_one_of_three.py`, then run the snapshot → see the tabletop pick-one-of-three layout

> Once you're happy, continue to generation below (generation is slow, so dial in the declaration in the snapshot first).

## 4. Why interaction needs collision-free planning

Simple pick: IK + a single straight line is enough. But once there is **environment interaction** — an articulated drawer, placing an object into a container, long-horizon multi-segment —
the arm must avoid the cabinet body and move under control during the contact phase, and **a straight line will inevitably collide**.

So the motion for `pick` / `place` is produced by **rocRobo collision-free planning** (running on ROCm);
grasp candidates come from a learned model, filtered by feasibility; the drawer's `open` / `close` uses a drag-handle primitive.
The generation step next will actually call this planning.

The **collision-blind vs collision-aware split-screen comparison** below (a pre-rendered artifact from rocRobo's `samples/demo_compare.py`,
shipped with the rocRobo image) shows it most intuitively: for the same goal, straight-line planning clips through / hits obstacles, while rocRobo detours to avoid them.

In [ ]:
from IPython.display import Video

# Pre-rendered comparison video from rocRobo samples/demo_compare.py, placed in videos/ as workshop material.
Video("workshop/videos/rocrobo_compare.mp4", embed=True, width=720)

## 5. Generate one interaction episode live (hero)

A single script call runs the full `open→pick→place→close`: rocRobo plans a collision-free trajectory for each segment,
the learned grasp picks a grasp, the goal point is re-anchored to the drawer's real-time opening, cameras render, and a LeRobot dataset + video are exported.

> CDNA3 has no graphics pipeline, so cameras use CPU software rendering; a single episode taking **tens of seconds to a few minutes** is normal. We only generate 1 episode live.

In [ ]:
import subprocess, sys, time

OUTPUT_DIR = "output/ws_drawer"
cmd = [
    sys.executable, "scripts/datagen/run_generated_scenario.py",
    "--scenario", "scenarios/pick_place_into_drawer.py",
    "--output-dir", OUTPUT_DIR,
    "--n-episodes", "1",
    "--seed", "42",
    "--grasp-planner", "auto",
    "--smoke",
]
print(">>", " ".join(cmd), "\n")
t0 = time.time()
subprocess.run(cmd, check=True)
print(f"\n[done] {time.time() - t0:.1f}s  ->  {OUTPUT_DIR}/")

## 6. Open the output

Play the episode video you just generated — the full `open→pick→place→close` plays out for you.
(If the live run didn't finish, fall back to a pre-generated video.)

In [ ]:
import glob
from IPython.display import Video

vids = sorted(glob.glob("output/ws_drawer/videos/*.mp4"))
if not vids:  # slow live run / didn't finish: fall back to pre-generated video
    vids = sorted(glob.glob("workshop/videos/*.mp4"))
print("video:", vids[0] if vids else "(none)")
Video(vids[0], embed=True, width=640) if vids else None

## 7. Swap a scenario and regenerate (declarative data factory)

The same pipeline, swap in a ~15-line declaration, and it produces different interaction data. This session ships three demo scenarios:

| scenario | task | type |
|---|---|---|
| `pick_place_into_drawer.py` | open drawer → grasp die → place in → close | long-horizon articulated (hero) |
| `pick_place_onto_supporter.py` | move an apple from the top shelf to the bottom shelf | collision-free pick-place (static) |
| `pick_one_of_three.py` | 3 objects placed randomly on a table, pick the apple | single-step rigid pick |

Change the `SCENARIO` below to pick one and rerun (you can also open a `.py` directly and edit a line — swap the object / move the drop point / change the success criterion).

In [ ]:
import subprocess, sys
from pathlib import Path

# Pick any demo scenario to regenerate:
#   scenarios/pick_one_of_three.py          tabletop pick-one-of-three (fastest)
#   scenarios/pick_place_onto_supporter.py  top->bottom collision-free pick-place
#   scenarios/pick_place_into_drawer.py     hero (same as §5)
SCENARIO = "scenarios/pick_one_of_three.py"

NAME = "ws_" + Path(SCENARIO).stem
cmd = [
    sys.executable, "scripts/datagen/run_generated_scenario.py",
    "--scenario", SCENARIO,
    "--output-dir", f"output/{NAME}",
    "--n-episodes", "1",
    "--seed", "1",
    "--grasp-planner", "auto",
    "--smoke",
]
print(">>", " ".join(cmd), "\n")
subprocess.run(cmd, check=True)

## 8. Summary

- **Where interaction-ready comes from**: articulated interaction (drawer joint) + long-horizon multi-segment + collision-free planning + contact semantics,
  all landing in a LeRobot dataset, ready to feed training directly.
- **The AMD / ROCm engineering story**: rocRecon (assets) / rocRobo (collision-free planning) / Genesis (physics + rendering)
  all run on ROCm, with clean licensing throughout.
- Change one line of the declaration to swap in a new task — this is a **declarative interaction data factory** running on AMD.